In [29]:
import tkinter as tk
from tkinter import ttk, messagebox
import requests
import pandas as pd
import logging
from requests.auth import HTTPBasicAuth
import urllib3
import threading
import re

# --------------------------------------------------
# Disable SSL Warnings
# --------------------------------------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --------------------------------------------------
# Logging
# --------------------------------------------------
logging.basicConfig(
    filename="jira_app.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# --------------------------------------------------
# Globals
# --------------------------------------------------
jira_url = None
session = None
return_to_extract_after_login = False  # ✅ NEW

CONFIG_FILE = "fields.xlsx"
OUTPUT_FILE = "output.xlsx"

# --------------------------------------------------
# Helper: Extract Field Values
# --------------------------------------------------
def extract_value(fields, jira_field, property_name):
    value = fields.get(jira_field)
    if value is None:
        return None

    if str(jira_field).lower() == "customfield_10000":
        sprint_names = []
        for sprint in value:
            match = re.search(r"name=([^,]+)", sprint)
            if match:
                sprint_names.append(match.group(1))
        return ", ".join(sprint_names) if sprint_names else None

    if isinstance(value, list) and all(isinstance(v, str) for v in value):
        return ", ".join(value)

    if not property_name or pd.isna(property_name):
        return value if not isinstance(value, (dict, list)) else None

    if isinstance(value, dict):
        return value.get(property_name)

    if isinstance(value, list):
        extracted = [
            str(item[property_name])
            for item in value
            if isinstance(item, dict) and property_name in item
        ]
        return ", ".join(extracted) if extracted else None

    return None

# --------------------------------------------------
# Parse Issues & Write Excel (File‑lock safe)
# --------------------------------------------------
def parse_issues_and_write_excel(issues):
    config_df = pd.read_excel(CONFIG_FILE)
    rows = []

    for issue in issues:
        row = {}
        fields = issue.get("fields", {})

        for _, cfg in config_df.iterrows():
            friendly = cfg["Friendly"]
            jira_field = cfg["Jira"]
            prop = cfg.get("Properties")

            if str(jira_field).lower() == "key":
                row[friendly] = issue.get("key")
            else:
                row[friendly] = extract_value(fields, jira_field, prop)

        rows.append(row)

    df = pd.DataFrame(rows)

    try:
        df.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")
        return True
    except PermissionError:
        root.after(
            0,
            lambda: messagebox.showwarning(
                "File In Use",
                "'output.xlsx' is currently open.\n\n"
                "Please close the file and click 'Extract Issues' again."
            )
        )
        return False

# --------------------------------------------------
# UI Helpers
# --------------------------------------------------
def clear_form():
    for widget in form_frame.winfo_children():
        widget.grid_forget()

# --------------------------------------------------
# Jira Connection
# --------------------------------------------------
def jira_connect_form():
    clear_form()

    tk.Label(form_frame, text="Jira URL:").grid(row=0, column=0, padx=10, pady=5)
    url_combobox.grid(row=0, column=1, padx=10, pady=5)
    url_combobox.current(0)

    tk.Label(form_frame, text="Username:").grid(row=1, column=0, padx=10, pady=5)
    username_entry.grid(row=1, column=1, padx=10, pady=5)

    tk.Label(form_frame, text="Password:").grid(row=2, column=0, padx=10, pady=5)
    password_entry.grid(row=2, column=1, padx=10, pady=5)

    connect_button.grid(row=3, columnspan=2, pady=10)

def connect_to_jira():
    global jira_url, session, return_to_extract_after_login

    jira_url = url_combobox.get().rstrip("/")
    username = username_entry.get()
    password = password_entry.get()

    session = requests.Session()
    session.auth = HTTPBasicAuth(username, password)
    session.verify = False

    try:
        r = session.get(f"{jira_url}/rest/api/2/myself")
        r.raise_for_status()
        messagebox.showinfo("Success", "Connected to Jira")

        clear_form()

        # ✅ Return user to Extract Issues if that was the intent
        if return_to_extract_after_login:
            return_to_extract_after_login = False
            create_extract_form()

    except Exception as e:
        messagebox.showerror("Connection Failed", str(e))

# --------------------------------------------------
# Extract Issues Form
# --------------------------------------------------
def create_extract_form():
    clear_form()

    tk.Label(form_frame, text="JQL to extract issues:").grid(
        row=0, column=0, sticky="w", padx=10
    )

    jql_text.grid(row=1, column=0, padx=10, pady=5)

    progress_bar.grid(row=2, column=0, padx=10, pady=5, sticky="ew")
    progress_bar.grid_remove()

    issue_extract_button.grid(row=3, column=0, pady=10)

# --------------------------------------------------
# Thread Orchestration (UPDATED FLOW)
# --------------------------------------------------
def start_extraction_thread():
    global return_to_extract_after_login

    # ✅ NEW: If not connected, redirect to Jira login
    if not session:
        return_to_extract_after_login = True
        jira_connect_form()
        return

    if not jql_text.get("1.0", tk.END).strip():
        messagebox.showerror("Error", "JQL cannot be empty")
        return

    progress_bar["value"] = 0
    progress_bar.grid()
    threading.Thread(target=background_worker, daemon=True).start()

def background_worker():
    try:
        issues = fetch_jira_issues_with_progress()
        success = parse_issues_and_write_excel(issues)

        if success:
            root.after(
                0,
                lambda: messagebox.showinfo(
                    "Success",
                    "output.xlsx created successfully.\n\nYou may now open the file."
                )
            )
    finally:
        root.after(0, reset_progress)
    clear_form()
# --------------------------------------------------
# Jira Fetch with Progress
# --------------------------------------------------
def fetch_jira_issues_with_progress():
    issues = []
    start_at = 0
    max_results = 500
    jql = jql_text.get("1.0", tk.END).strip()

    while True:
        response = session.get(
            f"{jira_url}/rest/api/2/search",
            params={
                "jql": jql,
                "startAt": start_at,
                "maxResults": max_results
            }
        )
        response.raise_for_status()
        data = response.json()

        total = data.get("total", 0)
        issues.extend(data.get("issues", []))

        percent = (len(issues) / total) * 100 if total else 100
        root.after(0, lambda v=percent: progress_bar.configure(value=v))

        if start_at + max_results >= total:
            break
        start_at += max_results

    return issues

def reset_progress():
    progress_bar.grid_remove()
    progress_bar["value"] = 0

# --------------------------------------------------
# App UI
# --------------------------------------------------
def exit_app():
    root.destroy()

root = tk.Tk()
root.title("Jira Automation")
root.geometry("450x360")

menu_bar = tk.Menu(root)

jira_menu = tk.Menu(menu_bar, tearoff=0)
jira_menu.add_command(label="Connect to Jira", command=jira_connect_form)
jira_menu.add_command(label="Extract Issues", command=create_extract_form)
jira_menu.add_command(label="Exit", command=exit_app)
menu_bar.add_cascade(label="Jira", menu=jira_menu)

root.config(menu=menu_bar)

form_frame = tk.Frame(root)
form_frame.grid(row=1, column=0, padx=10, pady=10)

url_combobox = ttk.Combobox(
    form_frame,
    values=[
        "https://jiraagile.sg.uobnet.com",
        "https://jiraagilep.sg.uobnet.com"
    ],
    width=35
)

username_entry = tk.Entry(form_frame, width=38)
password_entry = tk.Entry(form_frame, show="*", width=38)
connect_button = tk.Button(form_frame, text="Connect", command=connect_to_jira)

jql_text = tk.Text(form_frame, height=5, width=50, wrap="word")

progress_bar = ttk.Progressbar(form_frame, mode="determinate")

issue_extract_button = tk.Button(
    form_frame,
    text="Extract Issues",
    command=start_extraction_thread
)

root.mainloop()
